In [ ]:
# Corrigido: 'sklearn' -> 'scikit-learn' (o pacote 'sklearn' isolado está descontinuado)
!pip install -q awswrangler seaborn sweetviz pyarrow scikit-learn numpy xgboost

: 

# 0.0 Imports

In [ ]:
import gc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sweetviz as sv
import boto3
import awswrangler as wr
import sagemaker
import xgboost as xgb
import os
import pickle
import boto3
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Removidos: get_execution_role, get_image_uri, s3_input
# Esses imports do SDK antigo do SageMaker não são usados neste notebook
# (nao ha treino de modelo built-in da SageMaker aqui ainda).
# Se forem necessarios no futuro, usar as versoes atuais do SDK:
#   from sagemaker import image_uris
#   from sagemaker.inputs import TrainingInput
from sklearn.utils.class_weight import compute_sample_weight

# 0.1 - Helpers Functions

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

In [ ]:
# [DESABILITADO] Parquet ja salvo e consulta no Athena ja feita -- nao precisa
# rodar esta funcao de novo. Mantida so como referencia.
# def consultar_tabela_athena(database, tabela, chunksize=100_000):
    # """
    # Retorna um DataFrame com o resultado de um SELECT * na tabela do Athena informada,
    # lido em partes (chunks) para reduzir o pico de uso de memoria durante o download.

    # Responde a pergunta:
    # "Quais sao os dados brutos disponiveis nesta tabela do banco de dados?"

    # Parametros
    # ----------
    # database : str
        # Nome do database no Athena/Glue Catalog.
    # tabela : str
        # Nome da tabela dentro do database.
    # chunksize : int, opcional (padrao=100_000)
        # Numero de linhas por chunk durante o download. Reduz o pico de
        # memoria em instancias com pouca RAM (ex: ml.t3.medium).

    # Retorna
    # -------
    # df : pd.DataFrame
        # Resultado completo da query, concatenado a partir dos chunks.
    # """
    # ── 1. Configuracao de conexao ──────────────────────────────────────────
    # wr.config.aws_profile = "default"
    # wr.config.region = "us-east-1"

    # ── 2. Query ─────────────────────────────────────────────────────────────
    # query = f"SELECT * FROM {database}.{tabela}"
    # caminho_s3 = "s3://health-bridge-ggsolutions/athena-resultados/"

    # ── 3. Execucao em chunks ───────────────────────────────────────────────
    # ctas_approach=True: Athena entrega o resultado em formato colunar (Parquet)
    # internamente, mais leve de parsear que CSV.
    # chunks = wr.athena.read_sql_query(
        # query, database=database, s3_output=caminho_s3,
        # ctas_approach=True, chunksize=chunksize
    # )

    # ── 4. Concatenacao ─────────────────────────────────────────────────────
    # df = pd.concat(chunks, ignore_index=True)

    # return df

In [ ]:
# [DESABILITADO] Nao precisa reexecutar a consulta ao Athena -- o parquet ja esta salvo.
# database = 'database_health_bridge'
# tabela = 'table_train_gg_solutionstrain'

In [ ]:
# [DESABILITADO] Consulta ao Athena ja feita -- nao precisa reexecutar.
# df = consultar_tabela_athena(database, tabela)

In [ ]:
# [DESABILITADO] Depende de 'df' (consulta ao Athena), que nao sera reexecutada.
# df.head()

## 0.2 - Export to Parquet

In [ ]:
# [DESABILITADO] Exportacao ja feita -- train.parquet ja existe. Nao precisa reexecutar.
# Removido: df.to_csv('train2.csv') -- nunca era relido, so ocupava disco e tempo de escrita
# CSV de 1.3M x 76 colunas nao compensa quando o parquet ja resolve
# df.to_parquet('train.parquet', index=False)

# Guarda o shape original ANTES de liberar o df, para validar o parquet depois
# shape_original = df.shape
# print(f"Shape original (Athena): {shape_original}")

# del df
# gc.collect()

## 0.3 - Validar Parquet (sem recarregar tudo em memoria)

In [ ]:
# [DESABILITADO] Validacao ja feita quando o parquet foi gerado. So reexecute se quiser
# reconferir schema/shape (depende de 'shape_original', que so existe se a celula
# acima -- comentada -- for rodada de novo).
# import pyarrow.parquet as pq

# ParquetFile le apenas os metadados do arquivo, nao os dados em si
# parquet_file = pq.ParquetFile('train.parquet')

# n_linhas = parquet_file.metadata.num_rows
# n_colunas = parquet_file.metadata.num_columns

# print(f"Linhas no parquet: {n_linhas}")
# print(f"Colunas no parquet: {n_colunas}")

# if (n_linhas, n_colunas) == shape_original:
    # print("Parquet bate exatamente com o resultado do Athena.")
# else:
    # print(f"Divergencia! Original: {shape_original} | Parquet: {(n_linhas, n_colunas)}")

In [ ]:
# [DESABILITADO] Depende de 'parquet_file' (celula de validacao acima, comentada).
# Conferir schema (nomes e tipos de colunas)
# print(parquet_file.schema_arrow)

In [ ]:
# [DESABILITADO] Amostra rapida, opcional -- so para conferencia visual do parquet ja salvo.
# Amostra visual rapida (so algumas linhas, nao o arquivo inteiro)
# pd.read_parquet('train.parquet').head(5)

**Checagem cruzada (opcional, uma vez so):** rode `SELECT COUNT(*) FROM database_health_bridge.table_train_gg_solutionstrain`
direto no console do Athena e compare com `n_linhas` acima. Se baterem, o parquet esta 100% validado
e a celula do Athena (`consultar_tabela_athena`) nao precisa mais ser executada no resto do projeto.

## 0.4 - Import from Parquet

In [ ]:
df_parquet = pd.read_parquet('train.parquet')

In [ ]:
df_parquet.shape

# 1.0 - Data Description

In [ ]:
# Reducao do dataset o mais cedo possivel (logo na criacao do df1), para evitar que
# o pico de memoria de segurar o dataset completo + uma copia inteira ao mesmo tempo
# estoure os 8GB de RAM da ml.t3.large e derrube o kernel.
#
# Em vez de amostrar uma fracao fixa de TODAS as classes (o que reduziria tambem as
# classes minoritarias e forcaria mais reposicao/duplicacao no balanceamento da secao
# 2.1), aqui aplicamos um CAP por classe: so corta o excedente das classes que tem
# mais linhas que o cap, e preserva 100% das classes que ja sao pequenas -- que
# normalmente sao as classes clinicamente mais relevantes (ex.: obito, UTI).
CAP_POR_CLASSE = 60_000  # bem acima do alvo de 20k da secao 2.1, para manter
                          # diversidade real nas classes que ainda serao amostradas la

target_col = next(c for c in df_parquet.columns if c.lower() == 'classi_fin')

partes = []
for classe, grupo in df_parquet.groupby(target_col):
    if len(grupo) > CAP_POR_CLASSE:
        grupo = grupo.sample(CAP_POR_CLASSE, random_state=42)
    partes.append(grupo)

df1 = pd.concat(partes, ignore_index=True)
del partes

print(f'Dataset reduzido de {len(df_parquet):,} para {len(df1):,} linhas (cap de {CAP_POR_CLASSE:,}/classe).')

# Libera df_parquet: a partir daqui so trabalhamos com df1 em diante
del df_parquet
gc.collect()

In [ ]:
df1.info()

In [ ]:
df1.describe()

## 1.1 - Rename Columns

In [ ]:
df1.columns = df1.columns.str.lower()

## 1.2 - Data Dimensions

In [ ]:
print(f'Numero de linhas: {df1.shape[0]}')
print(f'Numero de colunas: {df1.shape[1]}')

## 1.3 - Fillout NA

In [ ]:
# soma vetorizada (custo baixo mesmo em 1.3M linhas x 76 colunas)
na_count = df1.isna().sum()
na_pct = (na_count / len(df1) * 100).round(2)

na_summary = pd.concat([na_count, na_pct], axis=1, keys=['na_count', 'na_pct'])
na_summary[na_summary['na_count'] > 0].sort_values('na_pct', ascending=False)

## 1.4 - Descriptive statistics

In [ ]:
# Separa atributos numericos e categoricos
num_attributes = df1.select_dtypes(include='number')
cat_attributes = df1.select_dtypes(exclude='number')

# agg() calcula todas as estatisticas em um unico passe por coluna (mais barato que
# chamar .mean(), .median(), .std() etc separadamente em 1.3M linhas)
num_stats = num_attributes.agg(['mean', 'median', 'std', 'min', 'max', 'skew', 'kurt']).T
num_stats['range'] = num_stats['max'] - num_stats['min']
print('Estatisticas - atributos numericos')
display(num_stats)

print('Estatisticas - atributos categoricos')
display(cat_attributes.describe().T)

# 2.0 - Statistical Analysis

In [ ]:
df2 = df1.copy()

del df1
gc.collect()

## 2.1 Random sample

In [ ]:
df2['classi_fin'].value_counts().sort_index(ascending=True)

In [ ]:
# Oversample / balanceamento por classe
# ATENCAO: a partir da secao 1.0, df1/df2 ja passou por um cap por classe (CAP_POR_CLASSE
# na celula de criacao do df1) -- classes minoritarias ficaram intactas, so as classes
# grandes tiveram o excedente cortado. A correlacao calculada abaixo reflete esse
# dataset (com o excedente das classes grandes ja removido), e nao o balanceamento
# artificial de 20000/classe feito neste bloco (que viria depois e distorceria ainda
# mais a correlacao real com o target).
#
# Corrigido: antes, 4 das 5 classes usavam .sample(20000, random_state=42) sem
# replace=True (undersampling, e quebra com ValueError se a classe tiver menos de
# 20000 linhas) e a classe 3 pegava todas as linhas sem limite (podendo dominar o
# dataset se tiver mais de 20000). O loop abaixo trata as 5 classes da mesma forma
# e funciona independente da distribuicao real de classi_fin.

In [ ]:
TARGET_POR_CLASSE = 20000

# Amostra cada classe para TARGET_POR_CLASSE linhas: sample sem reposicao se a classe
# tiver linhas suficientes, com reposicao (oversample real) caso contrario. Assim o
# balanceamento funciona independente da distribuicao real de classi_fin, sem quebrar
# com ValueError nem deixar uma classe dominando o resultado.
partes_balanceadas = []
for classe in sorted(df2['classi_fin'].unique()):
    subset = df2[df2['classi_fin'] == classe]
    reposicao = len(subset) < TARGET_POR_CLASSE
    partes_balanceadas.append(
        subset.sample(TARGET_POR_CLASSE, replace=reposicao, random_state=42)
    )

In [ ]:
df2_balanced = pd.concat(partes_balanceadas, ignore_index=True)

del partes_balanceadas
gc.collect()

In [ ]:
df2_balanced['classi_fin'].value_counts().sort_index(ascending=True)

In [ ]:
# Correlacao calculada no df2 (ja com cap por classe aplicado, ver nota na celula 2.1)
# -- ainda ANTES do balanceamento artificial de 20000/classe, entao reflete a
# distribuicao das classes so com o excedente das classes grandes removido, nao o
# dataset original inteiro nem a proporcao artificial de 20k/classe.
# corrwith evita montar a matriz de correlacao NxN inteira (mais leve na ml.t3.large,
# 2 vCPU / 8GB RAM): calcula so a correlacao de cada coluna numerica com o target.
# Colunas nao-numericas (ex.: 'hospital', se for string/categorica) ficam de fora
# dessa analise -- se forem relevantes, precisam de encoding antes de entrar aqui.
correlation = df2.select_dtypes('number').corrwith(df2['classi_fin'])
correlation.sort_values(ascending=False)

In [ ]:
df2_aux = df2_balanced[['classi_fin', 'raiox_res', 'fnt_in_cov', 'delta_uti', 'amostra', 'hospital']].copy()

# Corrigido: fillna(0) sem reatribuicao nao tinha efeito nenhum
df2_aux = df2_aux.fillna(0)

In [ ]:
my_report = sv.analyze(df2_aux)

# open_browser=False: a instancia do SageMaker nao tem navegador local, entao
# open_browser=True (padrao) tenta abrir um navegador e falha/trava. O relatorio
# ainda e salvo em HTML e pode ser aberto depois a partir do arquivo.
my_report.show_html(open_browser=False)

# 3.0 - Feature Engineering

In [ ]:
df3 = df2.copy()

In [ ]:
# Feature engineering: cria variaveis derivadas a partir de colunas do df2 que,
# sem transformacao, ou tem cardinalidade alta demais (sg_uf) ou nao capturam
# sozinhas padroes clinicos relevantes (idade continua, doencas separadas).

# 1) Faixa etaria (codigo ordinal): crianca, jovem, adulto, idoso.
faixas = [-1, 12, 18, 60, 200]
rotulos_faixa = ['crianca', 'jovem', 'adulto', 'idoso']
df3['faixa_etaria'] = pd.cut(df3['nu_idade_n'], bins=faixas, labels=rotulos_faixa)
df3['faixa_etaria_cod'] = df3['faixa_etaria'].cat.codes  # -1 = idade ausente/invalida

# 2) Score de comorbidades: quantos fatores de risco cronicos estao marcados
# como 1-Sim (NaN/9-Ignorado contam como "fator nao confirmado").
colunas_comorbidade = [
    'cardiopati', 'hematologi', 'sind_down', 'hepatica', 'asma', 'diabetes',
    'neurologic', 'pneumopati', 'imunodepre', 'renal', 'obesidade',
]
df3['score_comorbidades'] = (df3[colunas_comorbidade] == 1).sum(axis=1)

# 3) Macrorregiao a partir da UF: reduz cardinalidade de 27 estados para 5
# regioes, com um codigo ordinal simples (-1 = UF nao mapeada).
macro_regiao_por_uf = {
    'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte',
    'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste',
    'PB': 'Nordeste', 'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste',
    'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste',
    'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul',
}
df3['macro_regiao'] = df3['sg_uf'].map(macro_regiao_por_uf)
df3['macro_regiao_cod'] = df3['macro_regiao'].astype('category').cat.codes

# 4) Sazonalidade da semana epidemiologica (sem_pri), codificada em
# seno/cosseno para preservar a natureza ciclica (semana 52 fica perto da 1).
df3['sem_pri_sin'] = np.sin(2 * np.pi * df3['sem_pri'] / 52)
df3['sem_pri_cos'] = np.cos(2 * np.pi * df3['sem_pri'] / 52)

# 5) Score de esquema vacinal: quantos indicadores de vacinacao (gripe e
# COVID-19, incluindo doses) estao marcados como 1-Sim/aplicado.
colunas_vacina = ['vacina', 'vacina_cov', 'dose_1_cov', 'dose_2_cov', 'dose_ref']
df3['score_vacinal'] = (df3[colunas_vacina] == 1).sum(axis=1)

**Nota sobre `delta_uti` (adicionada apos analise exploratoria pre-producao):**

`delta_uti` e a feature de maior importancia do modelo treinado (maior `gain`,
`total_gain` e `cover` em `booster.get_score()`, acima de qualquer uma das
outras 10 features), mas o nome literal `delta_uti` **nao existe** no
dicionario oficial de dados do SIVEP-Gripe (conferido diretamente no PDF
oficial de 19/09/2022, opendatasus.saude.gov.br).

Rastreamento da origem: a coluna chega como passthrough bruto da tabela
`database_health_bridge.table_train_gg_solutionstrain` (Glue/Athena), sem
nenhuma coluna de data bruta na propria tabela que permita recalcular o valor.
O dicionario oficial define os campos correlatos da ficha:
- **53 - Internado em UTI?** (`UTI`): 1-Sim, 2-Nao, 9-Ignorado
- **54 - Data da entrada na UTI** (`DT_ENTUTI`)
- **55 - Data da saida da UTI** (`DT_SAIDUTI`)

A distribuicao empirica de `delta_uti` (consulta Athena: min -1, max 0-9
dependendo do grupo, media proxima de -1) e consistente com um delta curto de
dias entre essas datas, com `-1` funcionando como sentinela de
"nao aplicavel/sem UTI".

**Interpretacao de trabalho adotada pela equipe:** `delta_uti` = calculado a
partir de `DT_ENTUTI`/`DT_SAIDUTI`. Isso **nao e uma definicao oficial
publicada pelo DATASUS** com esse nome -- e a leitura mais consistente com as
evidencias disponiveis (dicionario oficial + distribuicao empirica), nao uma
confirmacao. Ainda assim, dado o quanto essa feature pesa nas previsoes,
qualquer decisao de trazer este modelo para um contexto clinico real deveria
confirmar a definicao exata com quem construiu o pipeline de dados.

In [ ]:
# Selecao final de colunas para o modelo: as 5 features originais (maior
# correlacao direta com classi_fin, ver secao 2.1) + as features derivadas
# criadas acima, que agregam sinal de idade, comorbidades, regiao, sazonalidade
# e vacinacao sem estourar a dimensionalidade.
#
# sintoma_resp_grave foi removida: corrwith (0.038) e mutual_info_classif
# (0.003) mostraram que ela nao discrimina entre classes -- esperado, ja que
# a base inteira e de notificacoes de SRAG, entao quase todo mundo ja tem
# sintoma respiratorio grave marcado.
colunas_modelo = [
    'id', 'classi_fin',
    'raiox_res', 'fnt_in_cov', 'delta_uti', 'amostra', 'hospital',
    'faixa_etaria_cod', 'score_comorbidades',
    'macro_regiao_cod', 'sem_pri_sin', 'sem_pri_cos', 'score_vacinal',
]
df3 = df3[colunas_modelo].copy()
df3 = df3.fillna(0)

In [ ]:
df3.describe()

In [ ]:
df3.info()

In [ ]:
# Checagem rapida: correlacao de cada feature (original + derivada na secao
# de feature engineering acima) com o target, para validar quais colunas
# realmente agregam sinal antes de treinar o modelo em vez de assumir que
# toda feature nova ajuda.
correlation_df3 = df3.select_dtypes('number').corrwith(df3['classi_fin'])
correlation_df3.sort_values(ascending=False)

In [ ]:
# Mutual information: complementa o corrwith acima. Mede dependencia (linear
# ou nao) entre cada feature e o target, e lida melhor com classi_fin como
# alvo categorico multiclasse -- a correlacao de Pearson (celula anterior)
# trata as classes como se fossem uma escala ordinal, o que elas nao sao.
from sklearn.feature_selection import mutual_info_classif

features_df3 = df3.drop(columns=['id', 'classi_fin']).astype('float64')

# sem_pri_sin/sem_pri_cos sao continuas; as demais sao discretas (codigos,
# contagens ou flags binarias) -- discrete_features precisa saber disso para
# estimar a informacao mutua corretamente.
colunas_continuas = ['sem_pri_sin', 'sem_pri_cos']
discrete_mask = [col not in colunas_continuas for col in features_df3.columns]

mi_scores = mutual_info_classif(
    features_df3, df3['classi_fin'],
    discrete_features=discrete_mask, random_state=42,
)
pd.Series(mi_scores, index=features_df3.columns).sort_values(ascending=False)

# 4.0 - EDA

In [ ]:
df4 = df3.copy()

# Base separada para a EDA completa: usamos df2 (antes da selecao de features
# da secao 3.0) pois as hipoteses abaixo dependem de colunas (vacina, area,
# contato com animais, idade, uf, semana epidemiologica) que nao sobrevivem
# a selecao de features feita para o modelo.
#
# Seleciona so as colunas usadas pelas hipoteses H1-H7 e pela secao 4.3 (em
# vez de copiar as 76 colunas de df2 inteiro) para nao duplicar memoria
# desnecessariamente -- importante na ml.t3.large (2 vCPU / 8GB RAM). Como
# bonus, o heatmap de correlacao da secao 4.5 fica legivel (em vez de uma
# matriz 76x76).
colunas_eda = [
    'classi_fin', 'vacina_cov', 'cs_zona', 'ave_suino', 'dispneia',
    'desc_resp', 'nu_idade_n', 'sg_uf', 'sem_pri', 'delta_uti',
]
df_eda = df2[colunas_eda].copy()

# Converte dtypes anulaveis do pandas (Int64, Float64, string) para os
# equivalentes numpy padrao. O seaborn (via matplotlib) nao lida bem com
# esses dtypes ao montar a legenda de graficos com hue numerico de muitos
# niveis (ex.: countplot com hue=classi_fin), lancando
# "TypeError: Cannot interpret 'Int64Dtype()' as a data type".
for col in df_eda.columns:
    dtype = df_eda[col].dtype
    if pd.api.types.is_extension_array_dtype(dtype):
        if pd.api.types.is_numeric_dtype(dtype):
            df_eda[col] = df_eda[col].astype('float64')
        else:
            df_eda[col] = df_eda[col].astype('object')

Pessoas que nao tomaram a vacina tem o maior numero de casos de covid

Pessoas que estao em area urbana a tendencia de ter covid é maior

Pessoas que trabalham com animais do tipo ave ou suino sempre tem alguma doença respiratoria

Pessoas com maior idade tem maior tendencia a ter doenças respiratorias

Pessoas que vivem no interior de santa catarina não tiveram nenhuma doença respiratoria

Pessoas que fumam não tiveram nenhuma doença respiratoria

Semanas do ano que são mais frias a quantidade de doenças respiratorias são maiores

    Isso vale para todos os estados?
Regioes onde a temperatura é mais alta tem um numero menor de doenças respiratorias

Pessoas que tiveram o exame aspectos de tomografia com probabilidade de ser covid, foi realmente covid?

Pessoas que tomaram antiviral tiveram menos sintomas graves

## 4.1 Overview e valores ausentes

In [ ]:
print(f'Linhas: {df_eda.shape[0]} | Colunas: {df_eda.shape[1]}')
df_eda.info()

In [ ]:
na_count = df_eda.isna().sum()
na_pct = (df_eda.isna().mean() * 100).round(2)
na_summary = pd.DataFrame({'qtd_na': na_count, 'pct_na': na_pct})
na_summary[na_summary['qtd_na'] > 0].sort_values('pct_na', ascending=False)

## 4.2 Analise univariada - variavel target (classi_fin)

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df_eda, x='classi_fin')
plt.title('Distribuicao de classi_fin (classificacao final do caso)')
plt.xlabel('classi_fin')
plt.ylabel('contagem')
plt.show()

df_eda['classi_fin'].value_counts(normalize=True).sort_index() * 100

## 4.3 Distribuicao das variaveis numericas

In [ ]:
numeric_cols = [c for c in ['nu_idade_n', 'delta_uti'] if c in df_eda.columns]

for col in numeric_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_eda[col].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f'Distribuicao de {col}')
    sns.boxplot(data=df_eda, x='classi_fin', y=col, ax=axes[1])
    axes[1].set_title(f'{col} por classi_fin')
    plt.tight_layout()
    plt.show()

## 4.4 Validacao das hipoteses

In [ ]:
def plot_categorical_vs_target(df, col, target='classi_fin'):
    if col not in df.columns:
        print(f'[aviso] coluna "{col}" nao encontrada no dataset - hipotese nao pode ser validada com os dados disponiveis.')
        return
    tab = pd.crosstab(df[col], df[target], normalize='index') * 100
    display(tab.round(1))
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=col, hue=target)
    plt.title(f'{col} vs {target}')
    plt.xticks(rotation=0)
    plt.legend(title='classi_fin', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

**H1 - Pessoas que nao tomaram a vacina (contra COVID-19) tem o maior numero de casos de COVID (classi_fin == 5)?**

In [ ]:
if 'vacina_cov' in df_eda.columns:
    covid_cases = df_eda[df_eda['classi_fin'] == 5]
    print(covid_cases['vacina_cov'].value_counts(normalize=True) * 100)
plot_categorical_vs_target(df_eda, 'vacina_cov')

**H2 - Pessoas em area urbana (cs_zona) tem maior tendencia a ter COVID?**

In [ ]:
plot_categorical_vs_target(df_eda, 'cs_zona')

**H3 - Pessoas que trabalham com aves/suinos (ave_suino) sempre tem alguma doenca respiratoria?**

Como o dataset nao possui uma coluna booleana de "doenca respiratoria", usamos
`classi_fin` (todo caso notificado e uma SRAG - Sindrome Respiratoria Aguda
Grave) e sintomas respiratorios (`dispneia`, `desc_resp`) como proxy.

In [ ]:
plot_categorical_vs_target(df_eda, 'ave_suino')

for sintoma in ['dispneia', 'desc_resp']:
    if sintoma in df_eda.columns and 'ave_suino' in df_eda.columns:
        tab = pd.crosstab(df_eda['ave_suino'], df_eda[sintoma], normalize='index') * 100
        print(f'--- {sintoma} por ave_suino (%) ---')
        display(tab.round(1))

**H4 - Pessoas com maior idade (nu_idade_n) tem maior tendencia a doencas respiratorias?**

In [ ]:
if 'nu_idade_n' in df_eda.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df_eda, x='classi_fin', y='nu_idade_n')
    plt.title('Idade por classi_fin')
    plt.show()
else:
    print('[aviso] coluna "nu_idade_n" nao encontrada.')

**H5 - Pessoas que vivem no interior de Santa Catarina nao tiveram nenhuma doenca respiratoria?**

Limitacao de dados: o dicionario so traz o codigo IBGE do municipio
(`id_municip` / `co_mun_not`), sem um mapeamento pronto capital x interior.
Validamos apenas em nivel de estado (`sg_uf` == 'SC') como aproximacao; para
validar por municipio seria necessario um de-para externo IBGE.

In [ ]:
if 'sg_uf' in df_eda.columns:
    df_sc = df_eda[df_eda['sg_uf'] == 'SC']
    print(f'Registros em SC: {len(df_sc)}')
    print(df_sc['classi_fin'].value_counts(normalize=True).sort_index() * 100)
else:
    print('[aviso] coluna "sg_uf" nao encontrada.')


**H6 - Pessoas que fumam nao tiveram nenhuma doenca respiratoria?**

Limitacao de dados: nao existe campo de tabagismo (fumante) no dicionario de
dados / dataset disponivel (SIVEP-Gripe). Esta hipotese **nao pode ser
validada** com as colunas atuais - seria necessario enriquecer a base com uma
fonte externa que traga esse dado.

**H7 - Semanas do ano mais frias tem maior quantidade de casos?**

In [ ]:
if 'sem_pri' in df_eda.columns:
    casos_por_semana = df_eda.groupby('sem_pri')['classi_fin'].count()
    plt.figure(figsize=(12, 4))
    casos_por_semana.plot(kind='line', marker='o')
    plt.title('Casos por semana epidemiologica dos primeiros sintomas (sem_pri)')
    plt.xlabel('Semana epidemiologica')
    plt.ylabel('Numero de casos')
    plt.show()
else:
    print('[aviso] coluna "sem_pri" nao encontrada.')

## 4.5 Correlacao entre variaveis numericas

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_eda.corr(numeric_only=True), cmap='coolwarm', center=0, annot=False)
plt.title('Matriz de correlacao - variaveis numericas')
plt.show()

## 4.6 Conclusoes da EDA

- A base esta desbalanceada entre classes de `classi_fin`; a secao 2.1 ja aplica
  undersampling nas classes majoritarias.
- Colunas com alto percentual de nulos (ver secao 4.1) precisam de tratamento
  (imputacao ou remocao) antes de entrarem no modelo.
- As hipoteses sobre vacina, zona urbana/rural e contato com animais podem ser
  checadas diretamente nos dados; idade e semana epidemiologica tambem.
- Hipoteses sobre tabagismo e localizacao (interior x capital) **nao podem ser
  totalmente validadas** com as colunas disponiveis - ver notas de limitacao
  acima.

## 4.7 Validacao das hipoteses (H1-H7)

Resumo das hipoteses levantadas na secao 4.4, confrontadas com os dados
apurados em cada bloco de analise acima.

| # | Hipotese | Resultado | Evidencia |
|---|----------|-----------|-----------|
| H1 | Nao vacinados contra covid tem o maior numero de casos de COVID (`classi_fin`==5) | **Parcialmente confirmada** | Entre os casos de COVID, 43.1% nao tomaram a vacina (`vacina_cov`==2) vs 38.1% que tomaram (`vacina_cov`==1) e 18.8% ignorado. O grupo nao vacinado e o maior, mas a margem sobre os vacinados e pequena. |
| H2 | Zona urbana (`cs_zona`) tem maior tendencia a COVID | **Refutada** | Entre os casos em zona urbana, 28.6% sao COVID; na zona rural a proporcao e maior (31.8%). Nao ha evidencia de maior tendencia na zona urbana. |
| H3 | Quem trabalha com aves/suinos (`ave_suino`) sempre tem alguma doenca respiratoria | **Refutada** | 77.8% desse grupo teve dispneia (nao 100%/"sempre"), e a diferenca para quem nao trabalha com aves/suinos (73.7%) e pequena. |
| H4 | Idade maior (`nu_idade_n`) implica maior tendencia a doencas respiratorias | **Nao avaliavel com estes dados** | O dataset so contem casos ja notificados de SRAG (sem grupo de controle sem a doenca), entao "tendencia" nao e mensuravel aqui. O boxplot mostra a classe 2 (outro virus respiratorio) concentrada em pacientes bem mais jovens (mediana ~8 anos); as demais classes ficam entre 50-70 anos, sem padrao crescente de idade x classe. |
| H5 | Pessoas no interior de SC nao tiveram nenhuma doenca respiratoria | **Refutada** | Validado a nivel de estado (limitacao de dado: sem de-para IBGE capital/interior). SC tem 9.497 registros, sendo 28.0% `classi_fin`==5 (COVID) e 21.4% `classi_fin`==4 - claramente ha casos. |
| H6 | Fumantes nao tiveram nenhuma doenca respiratoria | **Nao pode ser validada** | O SIVEP-Gripe/dataset disponivel nao tem coluna de tabagismo. |
| H7 | Semanas mais frias do ano tem mais casos | **Refutada / inconclusiva** | O pico de casos ocorre nas semanas 1-20 (verao/outono no Brasil), caindo ate a semana ~40 e subindo de novo no fim do ano - nao corresponde ao periodo mais frio (inverno, semanas ~24-39). O padrao parece refletir mais as ondas de COVID do que sazonalidade de frio.

# 5.0 Machine learning

In [ ]:
df5 = df4.copy()

# Normaliza dtypes anulaveis do pandas (Int64, Float64) para float64 padrao
# do numpy. df5 mistura colunas numpy puras (float64, int8) com colunas
# anulaveis vindas do awswrangler/feature engineering (delta_uti, classi_fin,
# sem_pri_sin/cos, score_vacinal) -- sem essa normalizacao, X_test.values
# pode virar um array dtype=object ao montar a amostra para o predictor,
# deixando a serializacao mais lenta e menos previsivel.
for col in df5.columns:
    if col != 'id' and pd.api.types.is_extension_array_dtype(df5[col].dtype):
        df5[col] = df5[col].astype('float64')

### 5.1 Filtrar classes invalidas de `classi_fin`

O dicionario oficial do SIVEP-Gripe define apenas 5 categorias validas para
`classi_fin` (1-Influenza, 2-Outro virus respiratorio, 3-Outro agente
etiologico, 4-Nao especificado, 5-COVID-19). O dataset tambem carrega os
codigos 6 (residual/legado, poucas dezenas de linhas) e 9 (Ignorado -
nao informado), que nao sao categorias clinicas reais e nao devem ser
aprendidas pelo modelo como se fossem classes validas. Filtramos essas
linhas antes do split treino/validacao/teste.

In [ ]:
antes = len(df5)
df5 = df5[df5['classi_fin'].isin([1, 2, 3, 4, 5])].reset_index(drop=True)
print(f'Linhas removidas (classi_fin 6/9/outros): {antes - len(df5):,}')
print(f'Linhas restantes: {len(df5):,}')
df5['classi_fin'].value_counts().sort_index()

In [ ]:
df5.sample(10)

In [ ]:
df5.columns

### 5.2 Criar bucket S3

In [ ]:
sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

# default_bucket() usa um nome gerado automaticamente por conta+regiao (ex.:
# sagemaker-us-east-1-<account_id>) e cria o bucket se ainda nao existir --
# evita o risco de colisao de um nome fixo (buckets S3 sao unicos globalmente
# entre TODAS as contas AWS, entao um nome fixo pode falhar com
# BucketAlreadyExists se outra conta ja tiver registrado esse nome).
bucket = sagemaker_session.default_bucket()
prefix1 = 'train'
prefix2 = 'test'
prefixest = 'est'

print(f'Bucket em uso: {bucket}')

### 5.3 Split treino / validacao / teste

Antes so havia um split 80/20 (treino/teste), e o proprio conjunto de teste
era reaproveitado como "validation" dentro do `xgb.train` -- ou seja, o
mesmo dado influenciava quando parar o treino e depois era usado pra
reportar a metrica final (nao e um teste sem vies).

Agora sao 3 conjuntos, estratificados por `classi_fin`:
- **Treino (70%)**: usado para ajustar o modelo.
- **Validacao (15%)**: monitorada a cada rodada do `xgb.train` e usada pelo
  `early_stopping_rounds` para decidir quando parar.
- **Teste (15%)**: fica intocado ate a avaliacao final (secao com
  `classification_report`/`confusion_matrix`) -- nunca influencia o treino
  nem o early stopping.

In [ ]:
X = df5.drop(['classi_fin', 'id'], axis=1)
y = df5['classi_fin']

# O algoritmo built-in XGBoost (objective multi:softmax) exige labels
# 0-indexadas (0..num_class-1). classi_fin vem como 1..5, entao remapeamos.
label_mapping = {label: idx for idx, label in enumerate(sorted(y.unique()))}
print('Mapeamento classi_fin original -> label do modelo:', label_mapping)
y = y.map(label_mapping)

# Split em treino/validacao/teste (70/15/15), estratificado por y.
# O conjunto de teste so e usado uma unica vez, no final, para reportar a
# metrica final sem vies -- nao entra no early stopping nem em nenhuma
# decisao durante o treino. E o que garante uma avaliacao nao-enviesada.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'Treino: {len(X_train):,} linhas')
print(f'Validacao: {len(X_val):,} linhas')
print(f'Teste: {len(X_test):,} linhas')

In [ ]:
# Peso por amostra (inversamente proporcional a frequencia da classe) para
# compensar o desbalanceamento sem duplicar linhas -- mais leve na RAM da
# ml.t3.large do que o oversample por replicacao, e aplicado somente ao
# treino: validacao e teste mantem a distribuicao real das classes.
#
# raiz quadrada do peso 'balanced' (em vez do peso cheio): o peso pleno
# supercorrige nas classes muito raras (ex.: classe 2, outro agente
# etiologico -- so 802 das 31140 linhas de teste), fazendo o modelo
# "chutar" essa classe em excesso (precision de 0.11 com o peso cheio).
# A raiz suaviza a diferenca entre pesos sem descartar o balanceamento.
balanced_weights = compute_sample_weight('balanced', y_train)
sample_weight_train = np.sqrt(balanced_weights)

In [ ]:
# O XGBoost built-in do SageMaker exige que a coluna target seja a PRIMEIRA
# coluna do CSV, sem header - antes o target era concatenado por ultimo.
train_data = pd.concat([y_train, X_train], axis=1)
test_data = pd.concat([y_test, X_test], axis=1)

In [ ]:
train_data.to_csv('train.csv', index=False, header=False)
test_data.to_csv('test.csv', index=False, header=False)

In [ ]:
# Fazer upload dos arquivos de treinamento e teste para o bucket S3
train_data_path = sagemaker_session.upload_data(path='train.csv', bucket=bucket, key_prefix=prefix1)
test_data_path = sagemaker_session.upload_data(path='test.csv', bucket=bucket, key_prefix=prefix2)

In [ ]:
print('Train data enviado para:', train_data_path)
print('Test data enviado para:', test_data_path)

In [ ]:
# distribution='FullyReplicated' explicito: o Local Mode monta a config do
# canal sem a chave 'S3DistributionType' quando isso nao e especificado,
# causando 'KeyError: S3DistributionType' na validacao do container
# built-in do XGBoost.
s3_input_train = sagemaker.inputs.TrainingInput(
    s3_data='s3://{}/{}/'.format(bucket, prefix1),
    content_type='text/csv',
    distribution='FullyReplicated',
)
s3_input_test = sagemaker.inputs.TrainingInput(
    s3_data='s3://{}/{}/'.format(bucket, prefix2),
    content_type='text/csv',
    distribution='FullyReplicated',
)

In [ ]:
print('s3_input_train:', s3_input_train.config['DataSource']['S3DataSource']['S3Uri'])
print('s3_input_test:', s3_input_test.config['DataSource']['S3DataSource']['S3Uri'])

In [ ]:
train_data['classi_fin'].value_counts()

In [ ]:
train_data.info()

## 5.4 Treino nativo com xgboost (sem Estimator/Local Mode)

O SageMaker Local Mode com o container built-in do XGBoost apresentou um bug
de compatibilidade (KeyError: 'S3DistributionType' na validacao do canal
dentro do container) que impede o treino de completar.

Como o objetivo final e servir o modelo via Lambda (secao 5.6) - que so
precisa de um model.pkl no S3, sem depender de nenhum SageMaker Endpoint -
a solucao mais simples e confiavel e treinar o XGBoost diretamente na
biblioteca xgboost, sem Estimator\container\Docker:

- Sem overhead de Docker (mais leve na RAM da ml.t3.large, que e
  compartilhada entre o kernel do Jupyter e qualquer container rodando).
- Sem esse bug do Local Mode (nao ha container, nao ha validacao de canal).
- Usa os mesmos hiperparametros (objective='multi:softmax', mesmo
  num_class) e os dados de treino/validacao/teste preparados na secao 5.3.
- `sample_weight_train` compensa o desbalanceamento das classes sem
  duplicar linhas (mais leve que oversample por replicacao).
- `early_stopping_rounds=10` monitora o conjunto de validacao (`dvalid`)
  e para o treino automaticamente quando ele para de melhorar -- o
  conjunto de teste (`dtest`) so aparece na avaliacao final.

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weight_train)
dvalid = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'multi:softmax',
    'num_class': len(df5['classi_fin'].unique()),
}

booster = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, 'train'), (dvalid, 'validation')],
    early_stopping_rounds=10,
)

## 5.5 Persistir o modelo treinado (pickle) para continuar na mesma instancia

Como o treino agora roda nativamente (sem container/Local Mode), o booster
ja fica em memoria ao final da celula anterior - nao ha artefato para baixar
do S3. Mesmo assim, vale salvar em disco, porque:

- Se o kernel desta notebook reiniciar (ou a notebook instance for
  parada/iniciada), a variavel booster em memoria some - sem o pickle,
  seria preciso rodar o treino de novo.
- O pickle tambem serve de "checkpoint": qualquer celula futura (inclusive
  o upload para o Lambda, secao 5.6) pode carregar o modelo direto do disco,
  sem depender do estado atual do kernel.

In [ ]:
# Diretorio no disco persistente da notebook instance (nao se perde ao
# reiniciar o kernel ou parar/iniciar a instancia - so se ela for deletada).
MODEL_DIR = '/home/ec2-user/SageMaker/modelo_treinado'
os.makedirs(MODEL_DIR, exist_ok=True)

pickle_path = os.path.join(MODEL_DIR, 'model.pkl')
with open(pickle_path, 'wb') as f:
    pickle.dump(booster, f)

print(f'Modelo (Booster) salvo localmente em: {pickle_path}')

### 5.5.1 Como recarregar o modelo depois (mesma instancia, sem retreinar)

Rode a celula abaixo em qualquer momento (mesmo apos reiniciar o kernel) para
recarregar o modelo ja treinado, sem precisar rodar `estimator.fit` de novo:

In [ ]:
pickle_path = '/home/ec2-user/SageMaker/modelo_treinado/model.pkl'
with open(pickle_path, 'rb') as f:
    booster_carregado = pickle.load(f)

booster_carregado

## 5.6 Inferencia via Lambda sem SageMaker Endpoint

Como o treino agora e nativo (sem Estimator), nao existe nenhum endpoint
criado neste notebook - a inferencia em producao e feita 100% pelo Lambda,
que carrega o model.pkl do S3 e roda a predicao ele mesmo. Isso evita
qualquer dependencia de cota de endpoint usage da conta.

Passos:
1. Sobe o model.pkl para o S3 (celula abaixo).
2. Builda uma imagem de container com o `xgboost-cpu` (mesma versao do
   treino, 3.2.0) e o `lambda_function.py`, e da push pra um repositorio
   ECR. Usamos imagem em vez de Lambda Layer porque o `xgboost` completo
   traz dependencias de GPU (`nvidia-nccl-cu12`) que passam muito do
   limite de 250 MB descompactados do formato zip/layer -- imagem de
   container aceita ate 10 GB.
3. Cria a funcao Lambda a partir dessa imagem, configurando as variaveis
   de ambiente MODEL_BUCKET/MODEL_KEY e memoria/timeout suficientes.
4. Da a role de execucao do Lambda permissao s3:GetObject no bucket/objeto
   do modelo.

In [ ]:
s3_client = sagemaker_session.boto_session.client('s3')
pickle_s3_key = 'modelos/model.pkl'
s3_client.upload_file(pickle_path, bucket, pickle_s3_key)
print(f'Modelo enviado para: s3://{bucket}/{pickle_s3_key}')

### 5.6.1 Codigo da funcao Lambda

A celula abaixo apenas **gera o arquivo `lambda_function.py`** nesta
notebook instance (o codigo em si roda no Lambda, nao aqui). Depois e so
zipar/colar esse arquivo na criacao da funcao.

**Atencao:** `event['instances']` precisa vir na MESMA ordem de colunas de
`X_train` (sem a coluna `id` nem o target `classi_fin`), do jeito que o
modelo foi treinado.

In [ ]:
lambda_code = """
import os
import json
import pickle
import boto3
import numpy as np
import xgboost as xgb

s3 = boto3.client('s3')

BUCKET = os.environ['MODEL_BUCKET']
KEY = os.environ['MODEL_KEY']
LOCAL_PATH = '/tmp/model.pkl'

# Ordem EXATA das colunas usadas no treino (X_train, secao 3.0 -- colunas_modelo
# sem 'id'/'classi_fin'). O Booster foi treinado com DMatrix construido a partir
# de um DataFrame, entao guarda esses feature_names internamente; um DMatrix sem
# nomes (numpy array puro) e rejeitado por booster.predict com
# "ValueError: data did not contain feature names".
FEATURE_NAMES = [
    'raiox_res', 'fnt_in_cov', 'delta_uti', 'amostra', 'hospital',
    'faixa_etaria_cod', 'score_comorbidades', 'macro_regiao_cod',
    'sem_pri_sin', 'sem_pri_cos', 'score_vacinal',
]

_booster = None


def _load_model():
    global _booster
    if _booster is None:
        s3.download_file(BUCKET, KEY, LOCAL_PATH)
        with open(LOCAL_PATH, 'rb') as f:
            _booster = pickle.load(f)
        # Forca softprob (probabilidade por classe) em vez do softmax original
        # (so classe vencedora) -- as arvores treinadas nao mudam, so a
        # transformacao final da saida. Nao exige retreino.
        _booster.set_param({'objective': 'multi:softprob'})
    return _booster


def lambda_handler(event, context):
    booster = _load_model()

    # event['instances']: lista de listas com as features, na MESMA ordem
    # de colunas usada no treino (X_train, sem 'id'/'classi_fin').
    dmatrix = xgb.DMatrix(np.array(event['instances']), feature_names=FEATURE_NAMES)

    # iteration_range restringe a predicao ao best_iteration do early
    # stopping (secao 5.4) -- sem isso, o predict usaria tambem as
    # rodadas extras de overfitting que o early_stopping_rounds deixou
    # treinadas antes de parar.
    best_iteration = getattr(booster, 'best_iteration', None)
    if best_iteration is not None:
        preds = booster.predict(dmatrix, iteration_range=(0, best_iteration + 1))
    else:
        preds = booster.predict(dmatrix)

    return {
        'statusCode': 200,
        'body': json.dumps({'predictions': preds.tolist()})
    }
"""

with open('lambda_function.py', 'w', encoding='utf-8') as f:
    f.write(lambda_code)

print('lambda_function.py gerado na pasta local do notebook.')

### 5.6.2 Build e push da imagem de container (ECR)

O pacote `xgboost` completo (o mesmo usado no treino) traz
`nvidia-nccl-cu12` como dependencia, usada so para treino distribuido em
GPU -- sozinha ja passa de 250 MB descompactada, o limite do Lambda para
zip/layers. Como so precisamos de inferencia em CPU, usamos `xgboost-cpu`
dentro da imagem (mesma versao major/minor/patch, 3.2.0, do pacote que
gerou o pickle -- o formato do Booster nao muda entre as duas variantes).

Empacotar como imagem de container em vez de zip+layer remove o limite de
250 MB (imagens aceitam ate 10 GB) e simplifica o Dockerfile: nao precisa
mais gerenciar Lambda Layer separada.

In [ ]:
ECR_REPO_NAME = 'requisicoes-ml-sagemaker'

sts_client = sagemaker_session.boto_session.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = sagemaker_session.boto_region_name

ecr_client = sagemaker_session.boto_session.client('ecr')
try:
    ecr_client.create_repository(repositoryName=ECR_REPO_NAME)
    print(f'Repositorio ECR criado: {ECR_REPO_NAME}')
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f'Repositorio ECR ja existia: {ECR_REPO_NAME}')

ecr_registry = f'{account_id}.dkr.ecr.{region}.amazonaws.com'
image_uri = f'{ecr_registry}/{ECR_REPO_NAME}:latest'
print(f'Imagem: {image_uri}')

In [ ]:
import subprocess

dockerfile_content = """FROM public.ecr.aws/lambda/python:3.12

# xgboost-cpu evita as dependencias de GPU (nvidia-nccl-cu12) do pacote
# xgboost completo -- sem elas o build fica mais rapido e a imagem bem
# menor, sem nenhuma perda para inferencia em CPU. Mesma versao do treino
# (3.2.0) para garantir compatibilidade do pickle do Booster.
RUN pip install --no-cache-dir xgboost-cpu==3.2.0

COPY lambda_function.py ${LAMBDA_TASK_ROOT}

CMD ["lambda_function.lambda_handler"]
"""

with open('Dockerfile', 'w', encoding='utf-8') as f:
    f.write(dockerfile_content)

print('Dockerfile gerado.')

subprocess.run(['docker', 'build', '-t', ECR_REPO_NAME, '.'], check=True)
subprocess.run(['docker', 'tag', f'{ECR_REPO_NAME}:latest', image_uri], check=True)

login_password = subprocess.run(
    ['aws', 'ecr', 'get-login-password', '--region', region],
    check=True, capture_output=True, text=True,
).stdout.strip()
subprocess.run(
    ['docker', 'login', '--username', 'AWS', '--password-stdin', ecr_registry],
    input=login_password, capture_output=True, text=True, check=True,
)

subprocess.run(['docker', 'push', image_uri], check=True)
print(f'Imagem publicada em: {image_uri}')

### 5.6.3 Criar (ou atualizar) a funcao Lambda

Usa a role de execucao ja existente (`requisicoes_ml_sagemaker-role-z4qp5v4e`)
e a imagem publicada no ECR acima (`PackageType='Image'` -- sem
Runtime/Handler/Layers, o handler ja esta definido no `CMD` do Dockerfile).

**Atencao:** ja existe uma Lambda criada manualmente no console, no fluxo
antigo do professor (`invoke_endpoint` num SageMaker Endpoint que nao existe
mais nesta conta) -- essa e do tipo `Zip`, nao `Image`. O `PackageType` de
uma funcao Lambda nao pode ser trocado depois de criada, entao se
`FUNCTION_NAME` ja existir como `Zip`, a celula abaixo apaga e recria como
`Image`. Se ja existir como `Image` (uma rodada anterior desta mesma
celula), so atualiza codigo/configuracao.

In [ ]:
FUNCTION_NAME = 'requisicoes_ml_sagemaker'
ROLE_NAME = 'requisicoes_ml_sagemaker-role-z4qp5v4e'

iam_client = sagemaker_session.boto_session.client('iam')
role_arn = iam_client.get_role(RoleName=ROLE_NAME)['Role']['Arn']
print(f'Role: {role_arn}')

lambda_client = sagemaker_session.boto_session.client('lambda')

common_env = {'Variables': {'MODEL_BUCKET': bucket, 'MODEL_KEY': pickle_s3_key}}


def _create_image_function():
    response = lambda_client.create_function(
        FunctionName=FUNCTION_NAME,
        PackageType='Image',
        Code={'ImageUri': image_uri},
        Role=role_arn,
        Timeout=30,
        MemorySize=1024,
        Environment=common_env,
    )
    lambda_client.get_waiter('function_active_v2').wait(FunctionName=FUNCTION_NAME)
    return response['FunctionArn']


try:
    function_arn = _create_image_function()
    print(f'Funcao Lambda criada: {function_arn}')
except lambda_client.exceptions.ResourceConflictException:
    existing = lambda_client.get_function(FunctionName=FUNCTION_NAME)
    if existing['Configuration']['PackageType'] != 'Image':
        print('Funcao existente e do tipo Zip (fluxo antigo) -- apagando para recriar como imagem.')
        lambda_client.delete_function(FunctionName=FUNCTION_NAME)
        function_arn = _create_image_function()
        print(f'Funcao Lambda recriada como imagem: {function_arn}')
    else:
        lambda_client.update_function_code(FunctionName=FUNCTION_NAME, ImageUri=image_uri)
        lambda_client.get_waiter('function_updated_v2').wait(FunctionName=FUNCTION_NAME)
        response = lambda_client.update_function_configuration(
            FunctionName=FUNCTION_NAME,
            Role=role_arn,
            Timeout=30,
            MemorySize=1024,
            Environment=common_env,
        )
        function_arn = response['FunctionArn']
        print(f'Funcao Lambda ja existia (imagem) -- codigo/configuracao atualizados: {function_arn}')

### 5.6.4 Teste de invocacao

Chama a funcao Lambda de verdade com algumas linhas do `X_test` (o mesmo
conjunto de teste da secao 5.7, nunca visto pelo treino) e compara a
predicao devolvida pelo Lambda com o rotulo real.

In [ ]:
import json

sample_size = 3
sample_instances = X_test.iloc[:sample_size].values.tolist()
sample_true = y_test.iloc[:sample_size].tolist()

invoke_response = lambda_client.invoke(
    FunctionName=FUNCTION_NAME,
    InvocationType='RequestResponse',
    Payload=json.dumps({'instances': sample_instances}),
)

if invoke_response.get('FunctionError'):
    print('ATENCAO -- Lambda retornou erro:')
    print(invoke_response['Payload'].read().decode('utf-8'))
else:
    payload = json.loads(invoke_response['Payload'].read())
    body = json.loads(payload['body'])
    predictions = body['predictions']
    print(f'Predicoes do Lambda (0-indexadas): {predictions}')
    print(f'Classes reais (y_test, 0-indexadas): {sample_true}')

## 5.7 Avaliacao no conjunto de teste completo

In [ ]:
# booster.predict com objective='multi:softmax' ja retorna a classe prevista
# diretamente (nao probabilidades) - sem precisar de endpoint/predictor.
# Avaliacao roda sobre dtest/y_test: o conjunto de teste que ficou de fora
# do treino e do early stopping (secao 5.3).
# iteration_range explicito garante que a predicao usa so ate a melhor
# rodada (best_iteration), descartando as rodadas extras de overfitting
# que o early_stopping_rounds deixou treinadas antes de parar -- independe
# de qual seja o comportamento padrao do predict() na versao do xgboost.
print('Melhor iteracao (early stopping):', booster.best_iteration)
y_pred = booster.predict(dtest, iteration_range=(0, booster.best_iteration + 1)).astype(int)
y_true = y_test.values

print('Acuracia no conjunto de teste completo:', accuracy_score(y_true, y_pred))
print()
print(classification_report(y_true, y_pred))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title('Matriz de confusao - conjunto de teste')
plt.show()

## 6.0 Conclusao

- O modelo XGBoost (multi:softmax, num_class = numero de classes de
  classi_fin) foi treinado nativamente com a biblioteca xgboost e
  avaliado no conjunto de teste completo (nao so numa amostra), com metricas
  de acuracia e classification report acima.
- O treino foi feito sem Estimator/container/Local Mode porque o Local Mode
  do SageMaker com o container built-in do XGBoost apresentou um bug de
  compatibilidade (KeyError: 'S3DistributionType') nesta conta/versao, e
  a conta e nova demais para ter cota de Training Job/Endpoint liberada.
- A inferencia em producao e feita via Lambda (secao 5.6), que carrega o
  model.pkl do S3 e roda a predicao diretamente - sem nenhum SageMaker
  Endpoint (real ou local) envolvido.
- O modelo treinado foi salvo em pickle (secao 5.5) e pode ser recarregado
  sem re-treinar.